# TerraAlert — ML Flood Detection (Prithvi EO) — Colab Runner

This notebook runs the full ML temporal flood detection pipeline on a GPU and produces:
- `outputs/flood_ml.tif` — uint8 flood mask (1=new flood, 0=land, 255=unusable)
- `outputs/water_pre.tif` — water mask from pre-event Sentinel-2
- `outputs/water_post.tif` — water mask from post-event Sentinel-2

**After running**: copy `flood_ml.tif`, `water_pre.tif`, `water_post.tif` from Colab back to `outputs/`.

---
## Model card summary (read before running)
| Parameter | Value |
|---|---|
| **Primary model** | `ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11` |
| **Fallback model** | `ibm-nasa-geospatial/Prithvi-EO-1.0-100M-sen1floods11` |
| **Band order** | BLUE(B2), GREEN(B3), RED(B4), NIR_NARROW(B8A), SWIR_1(B11), SWIR_2(B12) |
| **Normalisation** | Multiply raw DN by `0.0001` (constant_scale from config.yaml) |
| **Input tile size** | 512×512 px |
| **Output classes** | 2 (0=non-water, 1=water) |
| **terratorch** | ≥ 0.99 |
| **huggingface_hub** | ≥ 0.24 |


In [ ]:
# ── Step 0: Verify GPU ─────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Change Runtime → Runtime type → GPU.')
    print('The pipeline will fall back to CPU (very slow for 512×512 tiles).')


In [ ]:
# ── Step 1: Clone the repo (or mount Drive) ────────────────────────────────────
import os

REPO_URL = 'https://github.com/sahilkawankar-hub/Terraalert-Hackverse.git'
REPO_DIR = '/content/Terraalert-Hackverse'
BRANCH = 'ai'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -5


In [ ]:
# ── Step 2: Install dependencies ───────────────────────────────────────────────
!pip install -q -r requirements.txt
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q terratorch>=0.99 huggingface_hub>=0.24 tqdm
print('Dependencies installed.')


In [ ]:
# ── Step 3: Upload s2_pre.tif and s2_post.tif ──────────────────────────────────
# Option A: Upload from your local machine
from google.colab import files
import shutil, os

os.makedirs('outputs', exist_ok=True)

print('Upload s2_pre.tif and s2_post.tif (and perm_water.tif if available):')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join('outputs', fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved {fname} to {dest}')


In [ ]:
# ── Step 3 (Alternative B): Mount Google Drive ────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_OUTPUTS = '/content/drive/MyDrive/terraalert/outputs'
# import shutil, os
# for f in ['s2_pre.tif', 's2_post.tif', 'perm_water.tif']:
#     src = os.path.join(DRIVE_OUTPUTS, f)
#     if os.path.exists(src):
#         shutil.copy(src, os.path.join('outputs', f))
#         print(f'Copied {f}')


In [ ]:
# ── Step 4: Check required files exist ─────────────────────────────────────────
required = ['outputs/s2_pre.tif', 'outputs/s2_post.tif']
for f in required:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f'{"✓" if exists else "✗"} {f} ({size/1e6:.1f} MB)')

# perm_water.tif is needed for the pre/post logic; create dummy if missing
if not os.path.exists('outputs/perm_water.tif'):
    print('perm_water.tif not found — creating zero-filled dummy for Colab run.')
    import rasterio, numpy as np
    with rasterio.open('outputs/s2_pre.tif') as ref:
        profile = ref.profile.copy()
        H, W = ref.height, ref.width
    profile.update(count=1, dtype='uint8', nodata=255)
    with rasterio.open('outputs/perm_water.tif', 'w', **profile) as dst:
        dst.write(np.zeros((1, H, W), dtype=np.uint8))
    print('Created dummy perm_water.tif (all zeros — no permanent water excluded).')


In [ ]:
# ── Step 5: Run ML temporal flood detection ────────────────────────────────────
import sys
sys.path.insert(0, '/content/Terraalert-Hackverse')

from backend.ml.temporal import run_ml_detection

result = run_ml_detection(
    model_name='ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11',
    device='auto',  # Uses CUDA if available
    force=True,
)

print('\n=== ML Flood Detection Results ===')
if result.get('fallback_reason'):
    print(f'FALLBACK: {result["fallback_reason"]}')
else:
    print(f'Model used: {result["model_name"]}')
    print(f'Usable fraction: {result.get("ml_available_fraction", "N/A"):.1%}')
    print('Outputs: flood_ml.tif, water_pre.tif, water_post.tif')


In [ ]:
# ── Step 6 (Optional): Validate on Sen1Floods11 India chips ───────────────────
# Only if you have uploaded data/raw/S2Hand and data/raw/LabelHand

# from backend.ml.temporal import evaluate_ml_on_chips
# chip_result = evaluate_ml_on_chips(
#     model_name='ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11',
#     device='auto',
#     raw_dir=Path('/content/Terraalert-Hackverse/data/raw'),
#     max_chips=5,
# )
# if chip_result:
#     print(f'Chip validation: n={chip_result["n_chips"]}, '
#           f'IoU={chip_result["mean_iou"]:.3f}, F1={chip_result["mean_f1"]:.3f}')
#     print(chip_result['disclaimer'])


In [ ]:
# ── Step 7: Visualise results ──────────────────────────────────────────────────
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

outputs = [
    ('outputs/water_pre.tif', 'Water PRE-event', 'Blues'),
    ('outputs/water_post.tif', 'Water POST-event', 'Greens'),
    ('outputs/flood_ml.tif', 'Flood ML (new floods)', 'Reds'),
]

for ax, (path, title, cmap) in zip(axes, outputs):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
    data[data == 255] = np.nan  # mask nodata
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(title, fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('TerraAlert ML Flood Detection — Assam 2022', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/ml_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved outputs/ml_comparison.png')


In [ ]:
# ── Step 8: Download results ───────────────────────────────────────────────────
from google.colab import files
import os

for fname in ['flood_ml.tif', 'water_pre.tif', 'water_post.tif', 'ml_comparison.png']:
    fpath = os.path.join('outputs', fname)
    if os.path.exists(fpath):
        print(f'Downloading {fname} ({os.path.getsize(fpath)/1e6:.1f} MB) ...')
        files.download(fpath)
    else:
        print(f'NOT FOUND: {fname}')

print('\nDone! Copy downloaded files to your local outputs/ directory.')
print('Then run: python -m backend.fusion.fuse to merge with classical detection.')
